[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/AM/blob/main/06_Gradiente_Descendente_RegL.ipynb)


# Introdução ao Aprendizado de Máquina

**Professor: Diogo Ferreira de Lima Silva (TEP)**

**PPGEP - UFF**


Este notebook foi criado é com base em:

- Raschka, S., Liu, Y. H., Mirjalili, V., & Dzhulgakov, D. (2022). Machine Learning with PyTorch and Scikit-Learn: Develop machine learning and deep learning models with Python. Packt Publishing Ltd.

- Veja também o código do capítulo 9: https://github.com/rasbt/machine-learning-book



In [ ]:
### Bibliotecas Básicas
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt

# Gradiente Descendente em Regressão Linear

Nessa aula, vamos implementar o algoritmo gradiente descendente para regressão linear.

Para isso, utilizaremos os conceitos de Classes em Python.

## Implementando o Gradiente Descendente

Inicialmente, vamos criar uma classe vazia, sem nenhum método ou construtor.

In [ ]:
class LinearRegressionGD:
    pass

Em python, é comum que iniciemos as classes com um método especial, chamado usando o operador: __init__

Esse será o **construtor** de nossa classe. Assim, os parâmetros obtigatórios passados na criação de um objeto são definidos neste método.

Na construção da clase, um parâmetro especial chamado de **self** é sempre criado para se referir ao próprio objeto nas operações posteriores.

In [ ]:
class LinearRegressionGD:
    def __init__(self, eta=0.01, n_iter=50, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    pass

Agora, já poderíamos criar um objeto dessa classe. Os parâmetros definidos foram:

- eta = 0.01: a taxa de aprendizado
- n_iter = 50: o número de iterações do algoritmo
- random_state = 1: a semente aleatória que será utilizada na definição inicial de **w**

Os valores definidos dentro dos parênteses são usados como default caso nenhum parâmetro seja passado na criação do objeto.

In [ ]:
lr = LinearRegressionGD()
print(lr.eta)

In [ ]:
lr = LinearRegressionGD(eta = .005, n_iter = 20)
print(lr.eta)

Ok! Já temos uma classe. Porém, nada ainda pode ser feito.

Para estabelecermos o funcionamento de objetos dessa classe, criaremos os métodos. São funções criadas dentro das classes.

Iniciaremos com a função de aprendizado.

Seguindo o padrão da biblioteca sklearn, chamaremos esse método de **fit**.

In [ ]:
class LinearRegressionGD:
    def __init__(self, eta=0.01, n_iter=50, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    # Definição do nosso método!
    def fit(self, X, y):
        rgen = np.random.RandomState(self.random_state) # cria um gerador de números aleatórios
        self.w_ = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1]) # Cria o vetor inicial de pesos de forma aleatória
        self.b_ = np.array([0.]) # cria o bias inicial
        self.losses_ = [] # Uma lista vazia para alocarmos as perdas calculadas em cada passo do algoritmo

        # Loop no número de iterações
        for i in range(self.n_iter):
            output = np.dot(X, self.w_) + self.b_ # Realiza as previsões, dados os parâmetros w e b de momento
            desvios = (output -  y) # calcula o desvio de cada exemplo
            self.w_ -= self.eta * X.T.dot(desvios) / X.shape[0] # Atualiza o w
            self.b_ -= self.eta * desvios.mean() # Atualiza o b
            loss = (desvios**2).mean() # Computa a perda
            self.losses_.append(loss) # Coloca a perda visualizada no passo na lista de perdas
        return self # Retorna o objeto transformado (temos novos w_ e b_)

    pass

Agora, já somos capazes de treinar o nosso modelo

In [ ]:
data_rgen = np.random.RandomState(42)


# Vamos gerar dados genéricos
def generate_data(num_samples=1000):
    area = data_rgen.uniform(500, 3500, num_samples)
    quartos = data_rgen.randint(1, 6, num_samples)
    idade = data_rgen.randint(0, 80, num_samples)

    # Rótulo (preço do apartamento)
    base = 50000
    base_area = 100  # Preço adicional por m^2
    base_quarto = 20000  # Preço adicional por quarto
    base_idade = 500  # Redução no preço por ano
    noise = data_rgen.normal(0, 10000, num_samples)  # ruído aleatório

    price = (base + area * base_area + quartos * base_quarto - idade * base_idade + 0.05 * area**2 + 0.1 * area * idade + 50 * quartos**2 - 0.05 * idade*2 + noise)
    X=np.array([area,quartos,idade]).T
    y = price
    return X, y

X, y = generate_data()



In [ ]:
X

In [ ]:
y

In [ ]:
print(X.shape, y.shape)

## Dividindo os exemplos em conjuntos de treinamento e teste

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


### Treinamento do modelo

In [ ]:
lr = LinearRegressionGD()
lr.fit(X_train, y_train)

print(lr.w_, lr.b_)


In [ ]:
lr.losses_

**O modelo não está convergindo**. Provavelmente, a taxa de aprendizado precisaria ser bem menor.


Testem com outros valores!


Podemos também transformar os dados para ajudar na convergência do algoritmo.

Vamos utilizar um procedimento chamado StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
sc_x = StandardScaler()
sc_y = StandardScaler()

X_train_std = sc_x.fit_transform(X_train)
y_train_std = sc_y.fit_transform(y_train[:, np.newaxis]).flatten()


In [ ]:
lr = LinearRegressionGD()
lr.fit(X_train_std, y_train_std)

print(lr.w_, lr.b_)

In [ ]:
lr.losses_

Agora o algoritmo parece estar convergindo. Vamos treinar por mais tempo e plotar um gráfico

In [ ]:
lr = LinearRegressionGD(n_iter= 500)
lr.fit(X_train_std, y_train_std)


plt.plot(range(1, lr.n_iter+1), lr.losses_)
plt.ylabel('Erro Médio Quadrático')
plt.xlabel('Época')
plt.show()

### Previsão


Vamos usar nosso conjunto de teste para aplicar os valores que alcançamos no trenamento.

Precisamos incorporar mais um método à nossa classe. Mantendo a tradição das bibliotecas de ML, chamaremos de **predict**

In [ ]:
class LinearRegressionGD:
    def __init__(self, eta=0.01, n_iter=50, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def fit(self, X, y):
        rgen = np.random.RandomState(self.random_state) # cria um gerador de números aleatórios
        self.w_ = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1]) # Cria o vetor inicial de pesos
        self.b_ = np.array([0.]) # cria o bias inicial
        self.losses_ = [] # Uma lista vazia para alocarmos as perdas calculadas em cada passo do algoritmo

        # Loop no número de iterações
        for i in range(self.n_iter):
            output = np.dot(X, self.w_) + self.b_ # Realiza as previsões dado w e b de momento
            desvios = (output -  y) # calcula o desvio de cada exemplo
            self.w_ -= self.eta * X.T.dot(desvios) / X.shape[0] # Atualiza o w
            self.b_ -= self.eta * desvios.mean() # Atualiza o b
            loss = (desvios**2).mean() # Computa a perda
            self.losses_.append(loss) # Coloca a perda visualizada no passo na lista de perdas
        return self # Retorna o objeto transformado (temos novos w_ e b_)

    def predict(self, X):
        return np.dot(X, self.w_) + self.b_

In [ ]:
# Repetindo o treinamento
lr = LinearRegressionGD(n_iter= 500)
lr.fit(X_train_std, y_train_std)


# Transformando os dados de teste usando a mesma escala utilizada para os dados de treinamento
X_test_std = sc_x.transform(X_test)

# Aplicando a previsão
y_hat_std = lr.predict(X_test_std)

y_hat_std

Perceba que os dados que obtivemos não estão na escala de preços de apartamentos. Precisamos transformá-los.

Para isso, usamos a função inversa da transformação de y_train.

In [ ]:
y_hat = sc_y.inverse_transform(y_hat_std.reshape(-1, 1))

np.set_printoptions(formatter={'float': '{: 0.2f}'.format})

print(y_hat)

In [ ]:
y_test

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
print(f"w: {lr.w_}")
print(f"b: {lr.b_}")
print(f"MSE: {mean_squared_error(y_test, y_hat)}")
print(f"MAE: {mean_absolute_error(y_test, y_hat)}")

# Usando a biblioteca de Regressão Linear do Scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_std, y_train_std)
y_pred_std = lin_reg.predict(X_test_std)

y_pred = sc_y.inverse_transform(y_pred_std.reshape(-1, 1))

print(f"w: {lin_reg.coef_}")
print(f"b: {lin_reg.intercept_}")
print(f"MSE: {mean_squared_error(y_test, y_pred)}")
print(f"MAE: {mean_absolute_error(y_test, y_pred)}")

# Usando Transformações Polinomiais


Apesar de chamado de Regressão Linear, alguns ajustes nos dados podem deixar o modelo mais complexo, adicionando a possibilidade de não linearidade durante o treinamento.

Por exemplo, a função:

- $f(x) = w_1 x_1 + w_2 x_2 + b $


Poderia ser redefinida como:

- $f(x) = w_1 x_1 + w_2 x_2 + w_3 x_1^2 + w_4 x_2^2 + w_5 x_1 x_2 + b $


Perceba que o modelo continua linear do ponto de vista dos coeficientes. A não linearidade está nos atributos!

Em outras palavras, estamos criando "novos atributos" a partir dos atributos originais do conjunto de dados.




In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
poly_features = PolynomialFeatures(degree=2, include_bias=False)

X_poly_train = poly_features.fit_transform(X_train)
X_poly_test = poly_features.transform(X_test)


scaler_x = StandardScaler()
X_poly_train_std = scaler_x.fit_transform(X_poly_train)
X_poly_test_std = scaler_x.transform(X_poly_test)


lin_reg.fit(X_poly_train_std, y_train_std)
y_poly_pred_std = lin_reg.predict(X_poly_test_std)
y_poly_pred = sc_y.inverse_transform(y_poly_pred_std.reshape(-1, 1))

print(f"w: {lin_reg.coef_}")
print(f"b: {lin_reg.intercept_}")
print(f"MSE: {mean_squared_error(y_test, y_poly_pred)}")
print(f"MAE: {mean_absolute_error(y_test, y_poly_pred)}")

## Demonstração de Overfitting com Regressão Polinomial

Vamos criar um conjunto de dados sintético simples para ilustrar o conceito de *overfitting* quando o grau do polinômio na regressão se torna muito alto.

In [ ]:
# Gerando um conjunto de dados sintéticos
np.random.seed(20)
x = np.random.rand(20, 1) * 10  # 20 pontos entre 0 e 10
y = 2 * x**2 - 5 * x + 3 + np.random.randn(20, 1) * 10 # Função quadrática com ruído


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x, y, label='Dados Originais (com ruído)', s=20)

# Gerar pontos para plotar as curvas de regressão
x_plot = np.linspace(0, 10, 1000).reshape(-1, 1)

degrees_to_test = [1, 2, 5, 15] # Graus a serem testados
colors = ['green', 'orange', 'purple', 'red']

for i, degree in enumerate(degrees_to_test):
    # Transformar features para o grau polinomial atual
    poly_features = PolynomialFeatures(degree=degree, include_bias=False)
    x_poly = poly_features.fit_transform(x)
    x_poly_plot = poly_features.transform(x_plot)

    # Treinar o modelo de Regressão Linear
    model = LinearRegression()
    model.fit(x_poly, y)

    # Fazer previsões
    y_plot_pred = model.predict(x_poly_plot)

    # Plotar a curva de regressão com a nova legenda
    plt.plot(x_plot, y_plot_pred, color=colors[i], label=f'Grau {degree}')

plt.title('Regressão Polinomial com Diferentes Graus')
plt.xlabel('x')
plt.ylabel('y')
plt.ylim(-50, 180)
plt.legend()
plt.grid(True)
plt.show()